# Report Dataset Statistics

> Purpose: generate the dataset statistics used in the report.

In [1]:
from collections import Counter
from pathlib import Path

import pandas as pd

In [2]:
root = Path("../data/kaggle_speech_commands/tensorflow-speech-recognition-challenge")
if not (root / "train" / "audio").exists():
    candidates = list(Path("../data/kaggle_speech_commands").rglob("train/audio"))
    if not candidates:
        raise FileNotFoundError("Dataset root not found under ../data/kaggle_speech_commands")
    root = candidates[0].parent.parent

split_dir = root / "train" / "split_lists"
target_commands = {"yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"}


def norm_label(raw_label: str) -> str:
    if raw_label == "_background_noise_":
        return "__silence__"
    if raw_label in target_commands:
        return raw_label
    return "__unknown__"


split_files = [
    ("train_small", "small_training_list.txt", "Phases 1-3 train (10-class)"),
    ("valid_small", "small_validation_list.txt", "Phases 1-3 validation (10-class)"),
    ("test_small", "small_testing_list.txt", "Phases 1-3 diagnostics (10-class)"),
    ("train_extended", "extended_training_list.txt", "Phase 4 train (12-class)"),
    ("valid_extended", "extended_validation_list.txt", "Phase 4 validation (12-class)"),
    ("test_extended", "extended_testing_list.txt", "Phase 4 test (12-class)"),
]

rows = []
for split_name, file_name, used_for in split_files:
    path = split_dir / file_name
    if not path.exists():
        continue

    lines = [ln.strip() for ln in path.read_text(encoding="utf-8").splitlines() if ln.strip()]
    counts = Counter(norm_label(ln.split("/", 1)[0]) for ln in lines)

    rows.append(
        {
            "split": split_name,
            "total": int(sum(counts.values())),
            "classes_12": int(sum(1 for v in counts.values() if v > 0)),
            "command_total": int(sum(counts[label] for label in target_commands)),
            "unknown": int(counts["__unknown__"]),
            "silence": int(counts["__silence__"]),
            "used_for": used_for,
        }
    )

df_split_summary = pd.DataFrame(rows)
split_order = [name for name, _, _ in split_files]
df_split_summary["split"] = pd.Categorical(
    df_split_summary["split"], categories=split_order, ordered=True
)
df_split_summary = df_split_summary.sort_values("split").reset_index(drop=True)

print("Split summary (report table source):")
display(df_split_summary)

small = df_split_summary[df_split_summary["split"].str.contains("_small")]
extended = df_split_summary[df_split_summary["split"].str.contains("_extended")]

print("\nPolicy checks:")
print(
    "- Small splits are command-only:",
    bool((small[["unknown", "silence"]] == 0).all().all()),
)
print(
    "- Small split totals match 10x quotas (10000/2500/2500):",
    bool(
        small.set_index("split")["total"].to_dict()
        == {"train_small": 10000, "valid_small": 2500, "test_small": 2500}
    ),
)
print(
    "- Extended command and unknown counts are balanced per split:",
    bool((extended["command_total"] == extended["unknown"]).all()),
)

latex_table = df_split_summary[
    [
        "split",
        "total",
        "classes_12",
        "command_total",
        "unknown",
        "silence",
        "used_for",
    ]
].to_latex(index=False, escape=False)
print("\nLaTeX table snippet:")
print(latex_table)

Split summary (report table source):


,split,total,classes_12,command_total,unknown,silence,used_for
0,train_small,10000,10,10000,0,0,Phases 1-3 train (10-class)
1,valid_small,2500,10,2500,0,0,Phases 1-3 validation (10-class)
2,test_small,2500,10,2500,0,0,Phases 1-3 diagnostics (10-class)
3,train_extended,38249,12,18965,18965,319,Phase 4 train (12-class)
4,valid_extended,4733,12,2347,2347,39,Phase 4 validation (12-class)
5,test_extended,4780,12,2370,2370,40,Phase 4 test (12-class)



Policy checks:
- Small splits are command-only: True
- Small split totals match 10x quotas (10000/2500/2500): True
- Extended command and unknown counts are balanced per split: True

LaTeX table snippet:
\begin{tabular}{lrrrrrl}
\toprule
split & total & classes_12 & command_total & unknown & silence & used_for \\
\midrule
train_small & 10000 & 10 & 10000 & 0 & 0 & Phases 1-3 train (10-class) \\
valid_small & 2500 & 10 & 2500 & 0 & 0 & Phases 1-3 validation (10-class) \\
test_small & 2500 & 10 & 2500 & 0 & 0 & Phases 1-3 diagnostics (10-class) \\
train_extended & 38249 & 12 & 18965 & 18965 & 319 & Phase 4 train (12-class) \\
valid_extended & 4733 & 12 & 2347 & 2347 & 39 & Phase 4 validation (12-class) \\
test_extended & 4780 & 12 & 2370 & 2370 & 40 & Phase 4 test (12-class) \\
\bottomrule
\end{tabular}

